# Langchain

## Imports

In [1]:
!pip install langchain
!pip install langchain-openai
!pip install unstructured
!pip install faiss-cpu

In [2]:
# load our key
api_key = open("key.txt", "r").read().strip("\n")

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(openai_api_key=api_key)

In [4]:
llm.invoke("how can langsmith help with testing?")

AIMessage(content='Langsmith can help with testing by providing tools and frameworks for automated testing, such as unit testing, integration testing, and end-to-end testing. It can also assist in generating test cases, running tests, and analyzing test results. Additionally, Langsmith can help in setting up continuous integration and continuous deployment pipelines to automate the testing process and ensure that code changes do not introduce any issues.')

In [5]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are world class technical documentation writer."),
    ("user", "{input}")
])

from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

In [6]:
chain = prompt | llm | output_parser

In [7]:
chain.invoke({"input": "how can langsmith help with testing?"})

'Langsmith can help with testing in a variety of ways. Here are some ways in which Langsmith can assist with testing:\n\n1. Automation Testing: Langsmith can help automate the testing process by generating test scripts for various test cases. This can help save time and effort in executing repetitive test cases.\n\n2. Test Data Generation: Langsmith can assist in generating test data for testing various scenarios. This can help in ensuring that the application under test is thoroughly tested with different data sets.\n\n3. Code Coverage Analysis: Langsmith can provide code coverage analysis to identify areas of code that are not being tested. This can help in improving the overall test coverage of the application.\n\n4. Performance Testing: Langsmith can assist in performance testing by generating load on the application and measuring its performance under different load conditions.\n\n5. Integration Testing: Langsmith can help in testing the integration of different components of the 

## Retrieval Chain

In order to properly answer the original question ("how can langsmith help with testing?"), we need to provide additional context to the LLM. We can do this via retrieval. Retrieval is useful when you have too much data to pass to the LLM directly. You can then use a retriever to fetch only the most relevant pieces and pass those in.

In this process, we will look up relevant documents from a Retriever and then pass them into the prompt. A Retriever can be backed by anything - a SQL table, the internet, etc - but in this instance we will populate a vector store and use that as a retriever. For more information on vectorstores, see this documentation.

In [8]:
from langchain_community.document_loaders import DirectoryLoader

#loader = DirectoryLoader('../', glob="**/*.md")
loader = DirectoryLoader('documents', glob="**/*.md")
docs = loader.load()
len(docs)

10

Build Index

In [9]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(api_key=api_key)

In [10]:
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter()
documents = text_splitter.split_documents(docs)
vector = FAISS.from_documents(documents, embeddings)

Build Retrieval Chain

In [11]:
from langchain.chains.combine_documents import create_stuff_documents_chain

prompt = ChatPromptTemplate.from_template("""Answer the following question based only on the provided context:

<context>
{context}
</context>

Question: {input}""")

document_chain = create_stuff_documents_chain(llm, prompt)

In [12]:
from langchain.chains import create_retrieval_chain

retriever = vector.as_retriever()
retrieval_chain = create_retrieval_chain(retriever, document_chain)

In [13]:
response = retrieval_chain.invoke({"input": "Can you recommend me a recipe that contains chicken?"})
print(response["answer"])

Yes, I recommend trying the "Almond, Caper and Herb-Crusted Chicken Cutlets" recipe provided in the context. It is a Mediterranean-inspired dish that features chicken cutlets coated in a flavorful almond, caper, and herb mixture. Serve it with lemon wedges for a delicious and satisfying meal.


## Conversation Retrieval Chain

We can still use the create_retrieval_chain function, but we need to change two things:

The retrieval method should now not just work on the most recent input, but rather should take the whole history into account.
The final LLM chain should likewise take the whole history into account

In [14]:
from langchain.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder

# First we need a prompt that we can pass into an LLM to generate this search query

prompt = ChatPromptTemplate.from_messages([
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}"),
    ("user", "Given the above conversation, generate a search query to look up in order to get information relevant to the conversation")
])
retriever_chain = create_history_aware_retriever(llm, retriever, prompt)

In [15]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the user's questions based on the below context:\n\n{context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}"),
])
document_chain = create_stuff_documents_chain(llm, prompt)

retrieval_chain = create_retrieval_chain(retriever_chain, document_chain)

In [16]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = [HumanMessage(content="Can you recommend me a recipe that contains chicken?"), 
                AIMessage(content="Yes, you can try the \"Almond, Caper and Herb-Crusted Chicken Cutlets\" recipe provided in the context. It is a Mediterranean-style chicken dish that is quick and easy to prepare.")]
response = retrieval_chain.invoke({
    "chat_history": chat_history,
    "input": "Tell me how"
})

print(response["answer"])

To make the Almond, Caper, and Herb-Crusted Chicken Cutlets, you will need the following ingredients:

- ¾ cup sliced almonds, finely chopped
- ½ cup panko breadcrumbs
- Kosher salt and ground black pepper
- Four 4-ounce chicken cutlets (about ¼ inch thick)
- ¼ cup drained capers, finely chopped
- ¼ cup lightly packed fresh tarragon or fresh dill, chopped
- 6 tablespoons neutral oil
- Lemon wedges or Dijon mustard or sour cream, to serve

Here are the steps to prepare the dish:

1. Combine the almonds, panko, 1 teaspoon salt, and ½ teaspoon pepper in a bowl.
2. Season the chicken cutlets on both sides with salt and pepper, then sprinkle with the capers and tarragon, pressing to adhere.
3. Coat both sides of each cutlet with the almond mixture, pressing firmly.
4. Heat 3 tablespoons of oil in a skillet.
5. Add 2 cutlets and cook until golden brown on both sides.
6. Transfer the cooked cutlets to a paper towel-lined plate and wipe out the skillet. Repeat with the remaining cutlets and oi

## Agent

We've so far create examples of chains - where each step is known ahead of time. The final thing we will create is an agent - where the LLM decides what steps to take.

One of the first things to do when building an agent is to decide what tools it should have access to. For this example, we will give the agent access to two tools:

1. The retriever we just created. This will let it easily answer questions about LangSmith
2. A search tool. This will let it easily answer questions that require up to date information.